# Demystifying AI Agents: Build a Simple Agent From Scratch!

Welcome back, AI explorers! In our last post, we built **ChatGenie** — a cool terminal app that let us chat with an AI. It was fun, but it had a major limitation: the AI only knew what it was trained on. It couldn't look up your live database, check real-time prices, or look at your business files.

Today, we are fixing that by taking a massive leap into the world of **AI Agents**!

## What is an AI Agent? (The Hammer & Nail Analogy) 🛠️

Let's keep it real. Imagine you want to hang a beautiful picture frame, so you need to put a nail into the wall.

| Concept | Real World | AI Agent |
|---|---|---|
| **The Brain** | You thinking, *"I need to put this nail in the wall."* | The **LLM** — understands the goal |
| **The Tool** | You grab a **hammer** from the garage | Custom **Python functions** the agent can call |
| **Knowledge** | You remember where the studs in the wall are | Structured data (e.g., a product catalog) |
| **Memory** | You remember how hard to strike so you don't smash your thumb | Conversation history across turns |

An AI Agent works the exact same way! It combines an LLM **brain** with custom Python functions acting as its **tools**, structural data acting as its **knowledge**, and a history log acting as its **memory**.

## Step-by-Step Code Walkthrough 💻

Let's open up our notebook and build a **Product Query Agent** that uses a database tool to look up inventory prices and descriptions!

### Cell 1: Setting Up Our Environment

First, we import our agent framework components and load our API keys.

In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

# Load your secret API keys safely
load_dotenv()

True

### Cell 2: Creating Our Knowledge Base

Here is the store inventory data. This is the **Knowledge** our agent will draw from.

In [2]:
PRODUCTS = {
    "wireless headphones": {"price": 99.99, "description": "High-quality wireless headphones with noise cancellation."},
    "smartwatch": {"price": 199.99, "description": "Feature-rich smartwatch with fitness tracking and notifications."},
    "fitness tracker": {"price": 79.99, "description": "Compact fitness tracker with heart rate monitoring and sleep analysis."},
    "mechanical keyboard": {"price": 129.99, "description": "Durable mechanical keyboard with customizable RGB lighting."},
    "laptop stand": {"price": 49.99, "description": "Ergonomic laptop stand with adjustable height and angle."},
}

### Cell 3: Handing the AI a Tool 🧰

Now, let's build the hammer! In LangChain, we use a magic helper called a **decorator** (`@tool`). This tells the AI:

> *"Hey! Read this function's description. You are allowed to use this code whenever you need to find product details!"*

The `@tool` decorator transforms a standard Python function into a specialized plugin that AI agents can automatically understand, select, and invoke.

In [3]:
# Initialize our ultra-fast LLM brain
llm = ChatGroq(model="llama-3.1-8b-instant")


@tool
def get_product_info(product_name: str) -> str:
    """Get the price and description of a product.
    IMPORTANT: Provide ONLY the core product name. Do not include articles like 'the'."""

    # Clean up the user's input so it matches our dictionary keys
    clean_name = product_name.lower().replace("the ", "").strip(" .?")

    # Try to find an exact match
    product = PRODUCTS.get(clean_name)
    if product:
        return f"{clean_name.title()}: ${product['price']} - {product['description']}"

    # Smart fallback: look for partial matches (e.g., "tracker" instead of "fitness tracker")
    for key, product_data in PRODUCTS.items():
        if key in clean_name or clean_name in key:
            return f"{key.title()}: ${product_data['price']} - {product_data['description']}"

    return f"Sorry, we don't have information on '{product_name}'."

### Cell 4: Bringing the Agent to Life

Let's assemble our agent. We pass it the LLM brain, hand it our new tool, and give it clear instructions on how to format the final answer.

In [4]:
agent = create_agent(
    llm,
    tools=[get_product_info],
    system_prompt=(
        "You are a helpful product assistant. Use the provided tool to fetch product information. "
        "Always format your final response exactly like this: "
        "\n\n**[Product Name]**\nPrice: $[Price]\nDescription: [Description]\n\n"
        "DO NOT use any other tools, DO NOT make up information, and DO NOT output your internal reasoning."
    ),
)


def ask_agent(question: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)


# Test run!
ask_agent("Give me info about the tracker.")

**Fitness Tracker**
Price: $79.99
Description: Compact fitness tracker with heart rate monitoring and sleep analysis.


**Expected output:**

```
**Fitness Tracker**
Price: $79.99
Description: Compact fitness tracker with heart rate monitoring and sleep analysis.
```

**Boom!** The AI didn't know the price natively, so it chose to pick up our `get_product_info` tool, found the tracker, and formatted it beautifully!

---

## Leveling Up: Adding More Tools! 🛠️🛠️

What if we want to check customer reviews too? Let's add a second tool to our belt. Now the AI has to dynamically choose between the **hammer** or the **screwdriver** depending on what the user asks!

### Cell 5: The Reviews Dataset & Tool

In [5]:
REVIEWS = {
    "wireless headphones": {"average_rating": 4.5, "num_reviews": 1500},
    "smartwatch": {"average_rating": 4.2, "num_reviews": 2000},
    "fitness tracker": {"average_rating": 4.0, "num_reviews": 1200},
    "mechanical keyboard": {"average_rating": 4.7, "num_reviews": 800},
    "laptop stand": {"average_rating": 4.3, "num_reviews": 500},
}


@tool
def get_product_reviews(product_name: str) -> str:
    """Get the average rating and number of reviews for a product."""
    print(f"\n[Tool Debug] The LLM searched for exactly: '{product_name}'")

    clean_name = product_name.lower().replace("the ", "").strip(" .?")

    review = REVIEWS.get(clean_name)
    if review:
        return f"{clean_name.title()}: Average Rating: {review['average_rating']} stars from {review['num_reviews']} reviews."

    for key, review_data in REVIEWS.items():
        if key in clean_name or clean_name in key:
            return f"{key.title()}: Average Rating: {review_data['average_rating']} stars from {review_data['num_reviews']} reviews."

    return f"Sorry, we don't have review information on '{product_name}'."

### Cell 6: Updating the Agent Workspace

We give the agent access to both tools now and update our prompt.

In [6]:
agent = create_agent(
    llm,
    tools=[get_product_info, get_product_reviews],
    system_prompt=(
        "You are a helpful product assistant. Use the provided tools to fetch product information and reviews. "
        "Always format your final response exactly like this: "
        "\n\n**[Product Name]**\nPrice: $[Price]\nDescription: [Description]\n"
        "Average Rating: [Rating] stars from [Number of Reviews] reviews.\n\n"
        "DO NOT use any other tools, DO NOT make up information, and DO NOT output your internal reasoning."
    ),
)


def ask_agent(question: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)


ask_agent("Give me info about the fitness tracker.")

**Fitness Tracker**
Price: $79.99
Description: Compact fitness tracker with heart rate monitoring and sleep analysis.
Average Rating: N/A stars from N/A reviews.


---

## Uh Oh... The Agent Has Amnesia! 🧠💨

Watch what happens if we ask a natural follow-up question right after:

In [7]:
ask_agent("Shall I buy this?")

To help you make a decision, I'll need to know more about the product. Can you please provide the core product name?


**Expected output (something like):**

```
Sorry, I don't know what product you are talking about.
```

### Why did it fail?

Because by default, AI agents are **completely stateless**. Every time you run `agent.invoke()`, it's like the AI wakes up with total amnesia. It doesn't remember the last sentence it just said to you!

To answer questions like *"Should I buy this?"*, it needs **Memory**.

---

## Giving Our Agent a Memory Bank 🧠💾

To fix this, we use a utility called **`InMemorySaver`**. This acts like a continuous notebook that logs our conversation history using a unique tracker identifier called a **`thread_id`**.

### Cell 7: Building a Conversational Agent

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

agent2 = create_agent(
    llm,
    tools=[get_product_info, get_product_reviews],
    system_prompt=(
        "You are a helpful product assistant. Use the provided tools to fetch product information and reviews. "
        "Always format your final response exactly like this: "
        "\n\n**[Product Name]**\nPrice: $[Price]\nDescription: [Description]\n"
        "Average Rating: [Rating] stars from [Number of Reviews] reviews.\n\n"
    ),
    # We add the memory save state here!
    checkpointer=InMemorySaver(),
)


def ask_agent2(question: str) -> str:
    # Set a unique thread conversation session ID
    config = {"configurable": {"thread_id": "happy_shopping_session"}}
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]}, config=config)
    print(result["messages"][-1].content)

Now let's test a **continuous conversation stream**. Run these two cells in order — the second question relies on context from the first!

#### Question 1

In [9]:
ask_agent2("What is the price of the fitness tracker.")

**Fitness Tracker**
Price: $79.99
Description: Compact fitness tracker with heart rate monitoring and sleep analysis.
Average Rating: 4.5 stars from 2,115 reviews.


**Expected output:** Shows the Fitness Tracker price ($79.99)

#### Question 2 — the follow-up that needs memory!

In [10]:
ask_agent2("How are its reviews?")


[Tool Debug] The LLM searched for exactly: 'fitness tracker'
**Fitness Tracker**
Price: $79.99
Description: Compact fitness tracker with heart rate monitoring and sleep analysis.
Average Rating: 4.0 stars from 1,200 reviews.


**Expected output:**

```
[Tool Debug] The LLM searched for exactly: 'fitness tracker'

The Fitness Tracker has an average rating of 4.0 stars from 1200 reviews.
```

Notice how the agent remembered **"its"** referred to the fitness tracker from the previous turn — that's memory in action! 🎉

---

## Recap: What We Built

| Component | What it does |
|---|---|
| **LLM (ChatGroq)** | The brain — understands questions and decides what to do |
| **`@tool` functions** | The hands — fetch live data the LLM can't know on its own |
| **`PRODUCTS` / `REVIEWS`** | The knowledge base — structured data the tools query |
| **`create_agent()`** | Wires everything together with a system prompt |
| **`InMemorySaver` + `thread_id`** | Memory — lets the agent hold a multi-turn conversation |

In the next lesson, we'll explore more advanced agent patterns. Happy building! 🚀